# 04 — Weekly Team Strength

This notebook updates team strength sequentially using the new in season evidence created in `03_Inseason_Feature_Engineering.ipynb`.

The original preseason model remains unchanged.

The weekly model follows a sequential principle:

> The previous week's team strength is the prior. The most recently completed week provides new evidence.

For each target week, this notebook:

- Loads the frozen preseason ratings for reference
- Loads the previous week's team strength state
- Loads the target week's new in season feature table
- Applies the opponent adjusted weekly strength update
- Preserves separate offensive and defensive weekly signals
- Calculates current weekly rankings
- Compares rank and strength movement to the previous week
- Saves a separate weekly team strength file

No original preseason file or prior weekly file is overwritten.

In [15]:
from pathlib import Path

import numpy as np
import pandas as pd


## Paths and Weekly Settings

This notebook expects the output from:

`03_Inseason_Feature_Engineering.ipynb`

For Week 3:

- `TARGET_WEEK = 3`
- `PRIOR_WEEK = 2`

The Week 2 team strength file becomes the entering prior for Week 3.

In [16]:
PROJECT_ROOT = Path("../..")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
WEEKLY_DATA_DIR = PROCESSED_DIR / "weekly"

SEASON = 2026
TARGET_WEEK = 10
PRIOR_WEEK = TARGET_WEEK - 1

PRESEASON_STRENGTH_PATH = (
    PROCESSED_DIR
    / "2026_team_strength.parquet"
)

PRIOR_WEEK_STRENGTH_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{PRIOR_WEEK:02d}_sequential_team_strength.parquet"
)

INSEASON_FEATURES_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_inseason_features.parquet"
)

OUTPUT_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_sequential_team_strength.parquet"
)

print("Preseason ratings:", PRESEASON_STRENGTH_PATH)
print("Prior-week ratings:", PRIOR_WEEK_STRENGTH_PATH)
print("New in-season features:", INSEASON_FEATURES_PATH)
print("Weekly output:", OUTPUT_PATH)

Preseason ratings: ..\..\data\processed\2026_team_strength.parquet
Prior-week ratings: ..\..\data\processed\weekly\week_09_sequential_team_strength.parquet
New in-season features: ..\..\data\processed\weekly\week_10_inseason_features.parquet
Weekly output: ..\..\data\processed\weekly\week_10_sequential_team_strength.parquet


# Load Frozen Preseason Team Strength

The preseason team strength file is the final output of the original `05_Team_Strength_Model.ipynb`.

Its `team_strength` value is already expressed in point differential terms, so it can serve directly as the prior for weekly updates.

We immediately rename the important preseason columns so it is impossible to confuse the original ratings with the new weekly ratings.


In [17]:
preseason = pd.read_parquet(
    PRESEASON_STRENGTH_PATH
).copy()

required_preseason_columns = [
    "team",
    "team_strength",
    "baseline_team_strength",
    "personnel_adjustment",
    "personnel_strength",
    "roster_continuity",
    "roster_continuity_adjustment"
]

missing_preseason = [
    column
    for column in required_preseason_columns
    if column not in preseason.columns
]

if missing_preseason:
    raise KeyError(
        "Missing required preseason columns: "
        + ", ".join(missing_preseason)
    )

preseason = preseason.rename(
    columns={
        "strength_rank": "preseason_strength_rank",
        "team_strength": "preseason_team_strength",
        "baseline_team_strength": "preseason_baseline_team_strength",
        "personnel_adjustment": "preseason_personnel_adjustment",
        "personnel_strength": "preseason_personnel_strength",
        "roster_continuity": "preseason_roster_continuity",
        "roster_continuity_adjustment": "preseason_roster_continuity_adjustment"
    }
)

print("Preseason teams:", len(preseason))
print("Unique teams:", preseason["team"].nunique())

display(
    preseason[
        [
            "preseason_strength_rank",
            "team",
            "preseason_team_strength",
            "preseason_baseline_team_strength",
            "preseason_personnel_adjustment",
            "preseason_roster_continuity_adjustment"
        ]
    ].sort_values(
        "preseason_strength_rank"
    )
)


Preseason teams: 32
Unique teams: 32


,preseason_strength_rank,team,preseason_team_strength,preseason_baseline_team_strength,preseason_personnel_adjustment,preseason_roster_continuity_adjustment
0,1,BUF,4.816514,4.125764,0.934506,0.260091
1,2,LA,4.762452,3.257133,2.100811,0.489665
2,3,SEA,4.363421,3.515976,0.635683,0.932073
3,4,DET,4.018565,3.374644,1.041923,0.037533
4,5,DEN,3.569507,2.355154,0.979753,1.253003
5,6,BAL,3.074111,2.710864,0.636254,-0.037012
6,7,HOU,2.810133,2.193179,1.173833,-0.174691
7,8,GB,2.536317,1.718369,0.882628,0.576742
8,9,PHI,2.451919,2.324982,0.264690,-0.063756
9,10,SF,2.366788,1.805709,0.692939,0.290631


# Load Prior-Week Team Strength

The previous week's team strength file is the starting point for the new weekly update.

For Week 3, the model begins with the Week 2 ratings.

This prevents older games from being counted repeatedly and creates a true sequential rating system.

In [18]:
prior_week = pd.read_parquet(
    PRIOR_WEEK_STRENGTH_PATH
).copy()

required_prior_columns = [
    "team",
    "weekly_strength_rank",
    "weekly_team_strength",
    "cumulative_offense_adjustment",
    "cumulative_defense_adjustment"
]

missing_prior = [
    column
    for column in required_prior_columns
    if column not in prior_week.columns
]

if missing_prior:
    raise KeyError(
        "Missing required prior-week columns: "
        + ", ".join(missing_prior)
    )

prior_week = prior_week[
    required_prior_columns
].rename(
    columns={
        "weekly_strength_rank":
            "prior_week_strength_rank",
        "weekly_team_strength":
            "prior_week_team_strength",
        "cumulative_offense_adjustment":
            "prior_cumulative_offense_adjustment",
        "cumulative_defense_adjustment":
            "prior_cumulative_defense_adjustment"
    }
)

print(
    f"Week {PRIOR_WEEK} teams:",
    len(prior_week)
)

print(
    "Unique teams:",
    prior_week["team"].nunique()
)

display(
    prior_week.sort_values(
        "prior_week_strength_rank"
    )
)

Week 9 teams: 32
Unique teams: 32


,team,prior_week_strength_rank,prior_week_team_strength,prior_cumulative_offense_adjustment,prior_cumulative_defense_adjustment
0,BUF,1,6.296234,10.991284,-2.585323
1,JAX,2,4.547996,5.222063,7.695456
2,LA,3,4.376166,1.325544,2.320365
3,DEN,4,4.105679,-1.313456,3.431365
4,SF,5,3.931716,5.414943,3.313336
5,SEA,6,3.749348,0.459356,4.154178
6,CHI,7,3.577705,4.535805,3.730626
7,DET,8,3.225150,5.904386,-2.786507
8,MIN,9,2.861256,-3.502671,10.306436
9,PIT,10,2.844367,-0.929969,5.764852


# Load New Weekly In-Season Features

These features contain the new evidence generated in notebook `03`.

For Week 3, they describe each team's Week 2 performance relative to its entering Week 2 expectation.

The primary values used here are:

- `weekly_strength_update`
- `weekly_offense_adjustment`
- `weekly_defense_adjustment`

The overall strength update is opponent adjusted and uses diminishing returns for extreme single game results.

In [19]:
inseason = pd.read_parquet(
    INSEASON_FEATURES_PATH
).copy()

required_inseason_columns = [
    "team",
    "games_in_update",
    "entering_team_strength",
    "entering_strength_rank",
    "average_opponent_strength",
    "average_opponent_rank",
    "actual_margin",
    "expected_margin",
    "performance_residual",
    "weekly_strength_update",
    "weekly_offense_adjustment",
    "weekly_defense_adjustment"
]

missing_inseason = [
    column
    for column in required_inseason_columns
    if column not in inseason.columns
]

if missing_inseason:
    raise KeyError(
        "Missing required in-season columns: "
        + ", ".join(missing_inseason)
    )

print("In-season teams:", len(inseason))
print(
    "Unique teams:",
    inseason["team"].nunique()
)

display(
    inseason[
        required_inseason_columns
    ].sort_values(
        "weekly_strength_update",
        ascending=False
    )
)

In-season teams: 30
Unique teams: 30


,team,games_in_update,entering_team_strength,entering_strength_rank,average_opponent_strength,average_opponent_rank,actual_margin,expected_margin,performance_residual,weekly_strength_update,weekly_offense_adjustment,weekly_defense_adjustment
11,GB,1,2.104000,13,2.832965,11.0,18.0,-2.228965,20.228965,1.342028,3.521236,-0.608764
14,JAX,1,4.547996,2,-0.354664,17.0,23.0,3.402660,19.597340,1.327966,1.601700,1.821700
12,HOU,1,1.010378,16,-3.424708,26.0,22.0,2.935086,19.064914,1.315208,3.136573,-0.093427
3,BUF,1,6.296234,1,2.861256,9.0,17.0,1.934977,15.065023,1.187607,0.222297,2.542297
6,CIN,1,1.808836,14,-1.644763,20.0,17.0,1.953600,15.046400,1.186862,1.703321,0.723321
15,KC,1,2.693362,12,-5.190536,31.0,23.0,9.383898,13.616102,1.124753,1.720355,1.340355
27,SF,1,3.931716,5,-3.272264,24.0,21.0,8.703980,12.296020,1.058340,2.242290,0.662290
23,NYG,1,-2.376744,21,-3.331484,25.0,8.0,-0.545260,8.545260,0.816587,-0.609931,1.560069
26,SEA,1,3.749348,6,-4.337349,28.0,14.0,9.586697,4.413303,0.457789,-0.947651,2.722349
10,DET,1,3.225150,8,-4.601078,30.0,10.0,6.326227,3.673773,0.384826,-0.657540,1.812460


# Merge Prior Week State with New Evidence

The weekly model now combines three pieces of information:

1. Frozen preseason information for long term reference
2. Previous week's team strength as the current prior
3. Most recent week's opponent adjusted performance as new evidence

All 32 teams must remain in the table.

In [20]:
weekly_strength = (
    preseason
    .merge(
        prior_week,
        on="team",
        how="left",
        validate="one_to_one"
    )
    .merge(
        inseason,
        on="team",
        how="left",
        validate="one_to_one"
    )
)

weekly_fill_columns = [
    "games_in_update",
    "weekly_strength_update",
    "weekly_offense_adjustment",
    "weekly_defense_adjustment"
]

weekly_strength[
    weekly_fill_columns
] = (
    weekly_strength[
        weekly_fill_columns
    ]
    .fillna(0.0)
)

print("Merged teams:", len(weekly_strength))
print(
    "Unique teams:",
    weekly_strength["team"].nunique()
)

print(
    "Missing prior-week strengths:",
    weekly_strength[
        "prior_week_team_strength"
    ].isna().sum()
)

Merged teams: 32
Unique teams: 32
Missing prior-week strengths: 0


# Sequential Weekly Rating Update

The previous week's team strength is the starting point.

The current week's opponent adjusted performance update is then applied:

`weekly_team_strength = prior_week_team_strength + weekly_strength_update`

Therefore, for Week 3:

`Week 3 Strength = Week 2 Strength + Week 2 Performance Update`

Older results remain embedded in the prior week rating and are not counted again.

The offensive and defensive weekly adjustments remain available separately for the weekly scoring model.

In [21]:
weekly_strength["team_strength_change"] = (
    weekly_strength["weekly_strength_update"]
)

weekly_strength["weekly_team_strength"] = (
    weekly_strength["prior_week_team_strength"]
    + weekly_strength["team_strength_change"]
)

display(
    weekly_strength[
        [
            "team",
            "prior_week_team_strength",
            "weekly_offense_adjustment",
            "weekly_defense_adjustment",
            "team_strength_change",
            "weekly_team_strength"
        ]
    ].sort_values(
        "weekly_team_strength",
        ascending=False
    )
)

,team,prior_week_team_strength,weekly_offense_adjustment,weekly_defense_adjustment,team_strength_change,weekly_team_strength
0,BUF,6.296234,0.222297,2.542297,1.187607,7.483841
11,JAX,4.547996,1.601700,1.821700,1.327966,5.875962
9,SF,3.931716,2.242290,0.662290,1.058340,4.990056
1,LA,4.376166,-0.395084,1.324916,0.267286,4.643452
2,SEA,3.749348,-0.947651,2.722349,0.457789,4.207138
4,DEN,4.105679,0.421892,0.041892,-0.151823,3.953856
12,KC,2.693362,1.720355,1.340355,1.124753,3.818115
3,DET,3.225150,-0.657540,1.812460,0.384826,3.609975
18,CHI,3.577705,0.756171,-0.223829,-0.104808,3.472897
7,GB,2.104000,3.521236,-0.608764,1.342028,3.446027


In [22]:
weekly_strength["cumulative_offense_adjustment"] = (
    weekly_strength["prior_cumulative_offense_adjustment"]
    + weekly_strength["weekly_offense_adjustment"]
)

weekly_strength["cumulative_defense_adjustment"] = (
    weekly_strength["prior_cumulative_defense_adjustment"]
    + weekly_strength["weekly_defense_adjustment"]
)

assert weekly_strength[
    "cumulative_offense_adjustment"
].notna().all()

assert weekly_strength[
    "cumulative_defense_adjustment"
].notna().all()

# Updated Weekly Rankings

The current weekly rank is based on `weekly_team_strength`.

Ranking movement is now measured relative to the immediately previous week rather than the preseason ranking.

A positive `rank_change` means the team moved up.

For example:

- Week 2 rank = 8
- Week 3 rank = 5
- Rank change = +3

In [23]:
weekly_strength = weekly_strength.sort_values(
    [
        "weekly_team_strength",
        "prior_week_team_strength"
    ],
    ascending=[False, False]
).reset_index(drop=True)

weekly_strength["weekly_strength_rank"] = (
    np.arange(
        1,
        len(weekly_strength) + 1
    )
)

weekly_strength["rank_change"] = (
    weekly_strength["prior_week_strength_rank"]
    - weekly_strength["weekly_strength_rank"]
)

weekly_comparison_table = (
    weekly_strength[
        [
            "weekly_strength_rank",
            "team",
            "weekly_team_strength",
            "prior_week_strength_rank",
            "prior_week_team_strength",
            "team_strength_change",
            "rank_change"
        ]
    ]
    .copy()
)

display(weekly_comparison_table)

,weekly_strength_rank,team,weekly_team_strength,prior_week_strength_rank,prior_week_team_strength,team_strength_change,rank_change
0,1,BUF,7.483841,1,6.296234,1.187607,0
1,2,JAX,5.875962,2,4.547996,1.327966,0
2,3,SF,4.990056,5,3.931716,1.058340,2
3,4,LA,4.643452,3,4.376166,0.267286,-1
4,5,SEA,4.207138,6,3.749348,0.457789,1
5,6,DEN,3.953856,4,4.105679,-0.151823,-2
6,7,KC,3.818115,12,2.693362,1.124753,5
7,8,DET,3.609975,8,3.225150,0.384826,0
8,9,CHI,3.472897,7,3.577705,-0.104808,-2
9,10,GB,3.446027,13,2.104000,1.342028,3


# Biggest Weekly Movers

This table is diagnostic only.

It shows which teams experienced the largest overall rating changes from the previous week to the current week.

The underlying projection model still retains the full preseason, offense, defense, opponent and weekly information.

In [24]:
mover_columns = [
    "team",
    "prior_week_team_strength",
    "team_strength_change",
    "weekly_team_strength",
    "prior_week_strength_rank",
    "weekly_strength_rank",
    "rank_change"
]

biggest_risers = (
    weekly_strength[
        mover_columns
    ]
    .sort_values(
        "team_strength_change",
        ascending=False
    )
    .head(10)
)

biggest_fallers = (
    weekly_strength[
        mover_columns
    ]
    .sort_values(
        "team_strength_change",
        ascending=True
    )
    .head(10)
)

print("BIGGEST RISERS")
display(biggest_risers)

print()
print("BIGGEST FALLERS")
display(biggest_fallers)

BIGGEST RISERS


,team,prior_week_team_strength,team_strength_change,weekly_team_strength,prior_week_strength_rank,weekly_strength_rank,rank_change
9,GB,2.104000,1.342028,3.446027,13,10,3
1,JAX,4.547996,1.327966,5.875962,2,2,0
12,HOU,1.010378,1.315208,2.325586,16,13,3
0,BUF,6.296234,1.187607,7.483841,1,1,0
10,CIN,1.808836,1.186862,2.995699,14,11,3
6,KC,2.693362,1.124753,3.818115,12,7,5
2,SF,3.931716,1.058340,4.990056,5,3,2
17,NYG,-2.376744,0.816587,-1.560157,21,18,3
4,SEA,3.749348,0.457789,4.207138,6,5,1
7,DET,3.225150,0.384826,3.609975,8,8,0



BIGGEST FALLERS


,team,prior_week_team_strength,team_strength_change,weekly_team_strength,prior_week_strength_rank,weekly_strength_rank,rank_change
14,NE,2.832965,-1.342028,1.490937,11,15,-4
18,BAL,-0.354664,-1.327966,-1.682630,17,19,-2
27,LAC,-3.424708,-1.315208,-4.739916,26,28,-2
13,MIN,2.861256,-1.187607,1.673649,9,14,-5
21,ATL,-1.644763,-1.186862,-2.831625,20,22,-2
30,NYJ,-5.190536,-1.124753,-6.315289,31,31,0
26,LV,-3.272264,-1.058340,-4.330605,24,27,-3
25,PHI,-3.331484,-0.816587,-4.148071,25,26,-1
28,ARI,-4.337349,-0.457789,-4.795138,28,29,-1
29,MIA,-4.601078,-0.384826,-4.985904,30,30,0


# Update Diagnostics

These diagnostics check the size and distribution of the sequential weekly update.

The comparison is now:

`Prior Week Strength → Current Week Strength`

rather than:

`Preseason Strength → Current Week Strength`

In [25]:
movement = (
    weekly_strength[
        "team_strength_change"
    ].abs()
)

print(
    f"WEEK {PRIOR_WEEK} → "
    f"WEEK {TARGET_WEEK} TEAM-STRENGTH MOVEMENT"
)

print(
    weekly_strength[
        "team_strength_change"
    ]
    .describe()
    .round(3)
)

print()

print(
    "Mean absolute movement:",
    round(
        movement.mean(),
        3
    )
)

print(
    "Maximum absolute movement:",
    round(
        movement.max(),
        3
    )
)

print(
    "Team with largest movement:",
    weekly_strength.loc[
        movement.idxmax(),
        "team"
    ]
)

print()
print("RATING DISTRIBUTIONS")

distribution_check = pd.DataFrame(
    {
        "metric": [
            f"Week {PRIOR_WEEK} Team Strength",
            f"Week {TARGET_WEEK} Team Strength"
        ],
        "mean": [
            weekly_strength[
                "prior_week_team_strength"
            ].mean(),
            weekly_strength[
                "weekly_team_strength"
            ].mean()
        ],
        "std": [
            weekly_strength[
                "prior_week_team_strength"
            ].std(),
            weekly_strength[
                "weekly_team_strength"
            ].std()
        ],
        "min": [
            weekly_strength[
                "prior_week_team_strength"
            ].min(),
            weekly_strength[
                "weekly_team_strength"
            ].min()
        ],
        "max": [
            weekly_strength[
                "prior_week_team_strength"
            ].max(),
            weekly_strength[
                "weekly_team_strength"
            ].max()
        ]
    }
)

display(
    distribution_check.round(3)
)

WEEK 9 → WEEK 10 TEAM-STRENGTH MOVEMENT
count    32.000
mean      0.000
std       0.874
min      -1.342
25%      -0.547
50%       0.000
75%       0.547
max       1.342
Name: team_strength_change, dtype: float64

Mean absolute movement: 0.709
Maximum absolute movement: 1.342
Team with largest movement: GB

RATING DISTRIBUTIONS


,metric,mean,std,min,max
0,Week 9 Team Strength,0.002,3.563,-6.585,6.296
1,Week 10 Team Strength,0.002,4.037,-6.585,7.484


# Sanity Checks

These checks verify the sequential weekly update.

The weekly file should:

- contain exactly 32 unique teams
- preserve the frozen preseason information
- contain a valid prior week rating for every team
- contain no missing current weekly strength values
- have `team_strength_change` equal to the new opponent adjusted weekly update
- calculate current strength as prior week strength plus the weekly update
- never overwrite the preseason or prior week files

In [26]:
assert len(weekly_strength) == 32, (
    "Expected 32 teams in weekly strength table."
)

assert (
    weekly_strength["team"].nunique()
    == 32
), (
    "Expected 32 unique teams."
)

assert weekly_strength[
    "preseason_team_strength"
].notna().all(), (
    "Missing frozen preseason ratings."
)

assert weekly_strength[
    "prior_week_team_strength"
].notna().all(), (
    "Missing prior-week ratings."
)

assert weekly_strength[
    "weekly_team_strength"
].notna().all(), (
    "Missing current weekly ratings."
)

assert np.allclose(
    weekly_strength[
        "team_strength_change"
    ],
    weekly_strength[
        "weekly_strength_update"
    ]
), (
    "Team-strength change does not match "
    "the opponent-adjusted weekly update."
)

assert np.allclose(
    weekly_strength[
        "weekly_team_strength"
    ],
    (
        weekly_strength[
            "prior_week_team_strength"
        ]
        + weekly_strength[
            "team_strength_change"
        ]
    )
), (
    "Sequential weekly strength formula "
    "is inconsistent."
)

played_mask = weekly_strength["games_in_update"] > 0

assert np.allclose(
    weekly_strength.loc[
        played_mask, "entering_team_strength"
    ],
    weekly_strength.loc[
        played_mask, "prior_week_team_strength"
    ]
), (
    "Notebook 03 entering strength does not match "
    "the prior-week team strength."
)

bye_mask = ~played_mask

assert np.allclose(
    weekly_strength.loc[
        bye_mask, "weekly_team_strength"
    ],
    weekly_strength.loc[
        bye_mask, "prior_week_team_strength"
    ]
), "Bye-week teams did not retain their previous strength."

print(
    "All sequential weekly team-strength "
    "sanity checks passed."
)

All sequential weekly team-strength sanity checks passed.


# Final Weekly Team Strength Table

The saved file contains the full state needed by the weekly projection pipeline.

This includes:

- current weekly strength and rank
- previous week's strength and rank
- week to week movement
- frozen preseason information
- opponent adjusted performance information
- separate offensive and defensive weekly signals

The compact ranking table shown earlier is only for interpretation. The saved dataset retains the additional model information.

In [27]:
final_columns = [
    "weekly_strength_rank",
    "team",
    "weekly_team_strength",

    "prior_week_strength_rank",
    "prior_week_team_strength",
    "team_strength_change",
    "rank_change",

    "games_in_update",
    "average_opponent_rank",
    "average_opponent_strength",
    "actual_margin",
    "expected_margin",
    "performance_residual",
    "weekly_strength_update",

    "weekly_offense_adjustment",
    "weekly_defense_adjustment",
    "cumulative_offense_adjustment",
    "cumulative_defense_adjustment",

    "preseason_strength_rank",
    "preseason_team_strength",
    "preseason_baseline_team_strength",
    "preseason_personnel_adjustment",
    "preseason_personnel_strength",
    "preseason_roster_continuity",
    "preseason_roster_continuity_adjustment"
]

weekly_team_strength = (
    weekly_strength[
        final_columns
    ]
    .copy()
)

display(
    weekly_team_strength
)

,weekly_strength_rank,team,weekly_team_strength,prior_week_strength_rank,prior_week_team_strength,team_strength_change,rank_change,games_in_update,average_opponent_rank,average_opponent_strength,...,weekly_defense_adjustment,cumulative_offense_adjustment,cumulative_defense_adjustment,preseason_strength_rank,preseason_team_strength,preseason_baseline_team_strength,preseason_personnel_adjustment,preseason_personnel_strength,preseason_roster_continuity,preseason_roster_continuity_adjustment
0,1,BUF,7.483841,1,6.296234,1.187607,0,1.0,9.0,2.861256,...,2.542297,11.213581,-0.043026,1,4.816514,4.125764,0.934506,0.623004,0.573034,0.260091
1,2,JAX,5.875962,2,4.547996,1.327966,0,1.0,17.0,-0.354664,...,1.821700,6.823763,9.517156,12,1.713136,1.746953,-0.223729,-0.149153,0.563830,0.200841
2,3,SF,4.990056,5,3.931716,1.058340,2,1.0,24.0,-3.272264,...,0.662290,7.657233,3.975626,10,2.366788,1.805709,0.692939,0.461959,0.577778,0.290631
3,4,LA,4.643452,3,4.376166,0.267286,-1,1.0,19.0,-1.602244,...,1.324916,0.930460,3.645281,2,4.762452,3.257133,2.100811,1.400540,0.608696,0.489665
4,5,SEA,4.207138,6,3.749348,0.457789,1,1.0,28.0,-4.337349,...,2.722349,-0.488294,6.876527,3,4.363421,3.515976,0.635683,0.423788,0.677419,0.932073
5,6,DEN,3.953856,4,4.105679,-0.151823,-2,1.0,27.0,-3.816203,...,0.041892,-0.891564,3.473258,5,3.569507,2.355154,0.979753,0.653169,0.727273,1.253003
6,7,KC,3.818115,12,2.693362,1.124753,5,1.0,31.0,-5.190536,...,1.340355,2.829545,5.784367,13,1.302244,1.484935,-0.187743,-0.125162,0.510870,-0.140090
7,8,DET,3.609975,8,3.225150,0.384826,0,1.0,30.0,-4.601078,...,1.812460,5.246846,-0.974047,4,4.018565,3.374644,1.041923,0.694616,0.538462,0.037533
8,9,CHI,3.472897,7,3.577705,-0.104808,-2,1.0,15.0,1.097898,...,-0.223829,5.291976,3.506798,19,-0.051934,-0.069709,0.002783,0.001856,0.537634,0.032209
9,10,GB,3.446027,13,2.104000,1.342028,3,1.0,11.0,2.832965,...,-0.608764,1.215419,2.130240,8,2.536317,1.718369,0.882628,0.588419,0.622222,0.576742


# Save Weekly Team Strength

This writes only to:

`data/processed/weekly/`

The original:

`data/processed/2026_team_strength.parquet`

remains untouched.


In [28]:
OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

weekly_team_strength.to_parquet(
    OUTPUT_PATH,
    index=False
)

print("Saved:", OUTPUT_PATH)
print()
print(
    "Original preseason file remains:",
    PRESEASON_STRENGTH_PATH
)


Saved: ..\..\data\processed\weekly\week_10_sequential_team_strength.parquet

Original preseason file remains: ..\..\data\processed\2026_team_strength.parquet


# What This Means for Week 3

At the end of this notebook, every team has:

- its frozen preseason rating
- its Week 2 entering rating and rank
- its opponent adjusted Week 2 performance
- its Week 2 offensive and defensive signals
- its Week 2 → Week 3 rating movement
- its updated Week 3 team strength rating
- its updated Week 3 rank

The next stage will apply the Week 3 injury update and then use the resulting team state to generate Week 3 game projections.

The weekly ranking table is diagnostic only.

The saved weekly team strength file retains the additional information required by the projection pipeline.